# Phase 4.1 — one CPU-only cache ensemble (private)

Attach: (1) official competition data, (2) saved **Phase 2 private** output (including its retrieval and per-engine rerank caches), (3) saved **Phase 4 fixed** output (including `phase4_report.json`, `artifacts_phase4`, `submission_phase4_private_final.json` and config), (4) Phase 3 delta bundle (only for its offline project wheel). Internet Off; **CPU** accelerator. No models, GPU, retrieval, prepare, or rerank jobs are executed.

If and only if the ensemble beats *reproduced Phase 4 OOF* overall with no declining inner fold and exactly reconstructs the saved Phase 4 private answer sets, a single Phase 4.1 zip is created. Otherwise only a report is written; retain your Phase 4 submission. Saved Phase 4 outputs are never modified.


In [ ]:
import hashlib
import json
import os
import subprocess
import sys
from pathlib import Path

os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['CUDA_VISIBLE_DEVICES'] = ''
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'

INPUT_ROOT = Path('/kaggle/input')
WORK = Path('/kaggle/working/legalir-phase41-private')
RUNTIME = Path('/kaggle/working/legalir-phase41-runtime')
WORK.mkdir(parents=True, exist_ok=True)
RUNTIME.mkdir(parents=True, exist_ok=True)

def exactly_one(items, label):
    found = list(items)
    if len(found) != 1:
        raise RuntimeError(f'Expected exactly one {label}, found {len(found)}: {found}')
    return found[0]

def read(path):
    return json.loads(path.read_text(encoding='utf-8'))

REPORT_FILE = exactly_one(
    (p for p in INPUT_ROOT.rglob('phase4_report.json')
     if read(p).get('experiment_id') == 'phase4-phase2-legal-bge-protected-blender'
     and (p.parent / 'submission_phase4_private_final.json').is_file()),
    'saved Phase 4 output report',
)
PHASE4_OUTPUT = REPORT_FILE.parent
REPORT = read(REPORT_FILE)
private_matches = [
    p for p in INPUT_ROOT.rglob('private-official.json')
    if p.is_file() and hashlib.sha256(p.read_bytes()).hexdigest() == REPORT['test_sha256']
]
official_matches = [
    p for p in private_matches
    if (p.parent / 'train.json').is_file() and any(p.parent.rglob('context_*.json'))
]
PRIVATE_FILE = exactly_one(official_matches or private_matches, 'official private input matching Phase 4 report')
STATE_FILE = exactly_one(
    (p for p in INPUT_ROOT.rglob('inference_input_state.json')
     if (lambda s: s.get('filename') == 'private-official.json'
         and s.get('sha256') == REPORT['test_sha256']
         and s.get('questions') == REPORT['test_questions']
         and s.get('project_commit') == REPORT['project_commit'])(read(p))
     and (p.parent / 'artifacts_phase2_harrier').is_dir()),
    'matching saved Phase 2 private output',
)
PHASE2_ARTIFACTS = STATE_FILE.parent / 'artifacts_phase2_harrier'
DELTA_MANIFEST = exactly_one(
    (p for p in INPUT_ROOT.rglob('bundle_manifest.json')
     if read(p).get('experiment_id') == 'phase3-rerankers-harrier-retrieval'
     and read(p).get('project_commit') == REPORT['project_commit']),
    'Phase 3 delta bundle with matching project commit',
)
DELTA_ROOT = DELTA_MANIFEST.parents[1]
WHEEL = exactly_one((DELTA_ROOT / 'wheels').glob('uit_legalir-*.whl'), 'offline project wheel')
print('Phase 4 output:', PHASE4_OUTPUT)
print('Phase 2 caches:', PHASE2_ARTIFACTS)
print('Private input:', PRIVATE_FILE)
print('Project wheel:', WHEEL)


In [ ]:
# Only install the pinned project code; Kaggle's built-in NumPy, sklearn, PyYAML
# are used as-is. This step cannot fetch dependencies or models from the network.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps',
                '--target', str(RUNTIME), str(WHEEL)], check=True)
runtime_env = os.environ.copy()
runtime_env['PYTHONPATH'] = str(RUNTIME) + os.pathsep + str(WORK)
runtime_env['PYTHONNOUSERSITE'] = '1'
subprocess.run([sys.executable, '-c',
                'import legalir, sklearn, numpy, yaml; print("CPU dependencies OK", sklearn.__version__)'],
               check=True, env=runtime_env)


In [ ]:
# The exact Phase 4 feature code used by the completed fixed notebook.
(WORK / 'phase4_blend.py').write_text('from __future__ import annotations\n\nimport hashlib\nimport json\nimport pickle\nimport sys\nfrom collections import defaultdict\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport yaml\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\n\nfrom legalir.fusion import rrf\nfrom legalir.storage import read_json, read_jsonl, write_json\nfrom legalir.text import normalize_question\nfrom legalir.validation import grouped_folds, score_candidates, score_predictions, validate_submission_shape\n\n\nRERANKERS = ("jina", "vietnamese_reranker", "legal_reranker")\nRETRIEVAL_CHANNELS = (\n    "bm25",\n    "accent_char",\n    "vietlegal_harrier",\n    "vietnamese_embedding",\n    "nemotron",\n    "query_memory",\n    "query_exact",\n)\n\n\ndef fuse_cached(\n    retrievals: dict[str, dict[str, Any]],\n    weights: dict[str, float],\n    rrf_k: int,\n    limit: int,\n) -> dict[str, dict[str, list[str]]]:\n    return {\n        qid: {"candidates": rrf(retrieval["channels"], weights, rrf_k, limit)}\n        for qid, retrieval in retrievals.items()\n    }\n\n\ndef engine_ranks(artifacts: Path, split: str, engine: str, fold: int | None = None) -> dict[str, list[str]]:\n    suffix = f"_{fold}" if fold is not None else ""\n    separate = artifacts / f"rerank_{split}{suffix}_{engine}.json"\n    if separate.is_file():\n        return read_json(separate)[engine]\n    combined = artifacts / f"rerank_{split}{suffix}.json"\n    payload = read_json(combined)\n    if engine not in payload:\n        raise RuntimeError(f"{engine} is missing from {combined}")\n    return payload[engine]\n\n\ndef inner_fold(question: str) -> int:\n    # A salt different from grouped_folds is essential: every selected question\n    # is already in outer fold zero under the main fold hash.\n    key = "phase4-inner-v1:" + normalize_question(question)\n    return int(hashlib.sha256(key.encode("utf-8")).hexdigest()[:12], 16) % 5\n\n\ndef reciprocal_rank(rank: int, k: int = 20) -> float:\n    return 1.0 / (k + rank)\n\n\ndef rank_features(rank: int, missing_rank: int) -> list[float]:\n    clipped = min(rank, missing_rank)\n    denominator = max(1, missing_rank - 1)\n    return [\n        reciprocal_rank(clipped),\n        1.0 / clipped,\n        1.0 - min(clipped - 1, denominator) / denominator,\n        float(clipped <= 5),\n        float(clipped <= 10),\n        float(clipped <= 20),\n        float(clipped <= 50),\n        float(clipped < missing_rank),\n    ]\n\n\ndef feature_vector(document: str, orders: dict[str, dict[str, int]]) -> list[float]:\n    features: list[float] = []\n    core_ranks: list[int] = []\n    for name in ("first_stage", *RERANKERS):\n        rank = orders[name].get(document, 81)\n        core_ranks.append(rank)\n        features.extend(rank_features(rank, 81))\n    retrieval_ranks: list[int] = []\n    for name in RETRIEVAL_CHANNELS:\n        rank = orders[name].get(document, 151)\n        retrieval_ranks.append(rank)\n        features.extend(rank_features(rank, 151))\n\n    all_ranks = core_ranks + retrieval_ranks\n    features.extend(\n        [\n            float(sum(rank <= 5 for rank in core_ranks)),\n            float(sum(rank <= 10 for rank in core_ranks)),\n            float(sum(rank <= 20 for rank in core_ranks)),\n            float(sum(rank <= 20 for rank in retrieval_ranks)),\n            float(sum(rank < 151 for rank in retrieval_ranks)),\n            float(min(all_ranks)),\n            float(max(core_ranks)),\n            float(np.mean(core_ranks)),\n            float(np.std(core_ranks)),\n        ]\n    )\n    phase2_score = (\n        0.3 * reciprocal_rank(core_ranks[0])\n        + 0.5 * reciprocal_rank(core_ranks[1])\n        + 0.5 * reciprocal_rank(core_ranks[2])\n    )\n    legal_residual = reciprocal_rank(core_ranks[3]) - reciprocal_rank(core_ranks[0])\n    features.extend([phase2_score, legal_residual])\n    return features\n\n\ndef make_rows(\n    questions: list[dict[str, Any]],\n    fused: dict[str, dict[str, Any]],\n    retrievals: dict[str, dict[str, Any]],\n    rerankings: dict[str, dict[str, list[str]]],\n    labelled: bool,\n) -> tuple[np.ndarray, np.ndarray, list[tuple[str, str]], np.ndarray]:\n    rows: list[list[float]] = []\n    labels: list[int] = []\n    metadata: list[tuple[str, str]] = []\n    groups: list[int] = []\n    for question in questions:\n        qid = question["qid"]\n        candidates = fused[qid]["candidates"][:80]\n        orders = {"first_stage": {doc: rank for rank, doc in enumerate(candidates, 1)}}\n        for name, values in rerankings.items():\n            orders[name] = {doc: rank for rank, doc in enumerate(values[qid], 1)}\n        for name in RETRIEVAL_CHANNELS:\n            orders[name] = {doc: rank for rank, doc in enumerate(retrievals[qid]["channels"][name], 1)}\n        truth = set(question.get("answers", []))\n        group = inner_fold(question["question"])\n        for document in candidates:\n            rows.append(feature_vector(document, orders))\n            labels.append(int(document in truth) if labelled else 0)\n            metadata.append((qid, document))\n            groups.append(group)\n    return (\n        np.asarray(rows, dtype=np.float32),\n        np.asarray(labels, dtype=np.int8),\n        metadata,\n        np.asarray(groups, dtype=np.int8),\n    )\n\n\ndef standardize_per_query(values: np.ndarray, metadata: list[tuple[str, str]]) -> np.ndarray:\n    result = np.zeros(len(values), dtype=np.float64)\n    by_qid: defaultdict[str, list[int]] = defaultdict(list)\n    for index, (qid, _) in enumerate(metadata):\n        by_qid[qid].append(index)\n    for indices in by_qid.values():\n        scores = values[indices]\n        scale = scores.std()\n        result[indices] = (scores - scores.mean()) / (scale if scale > 1e-9 else 1.0)\n    return result\n\n\ndef scores_by_question(values: np.ndarray, metadata: list[tuple[str, str]]) -> dict[str, dict[str, float]]:\n    output: defaultdict[str, dict[str, float]] = defaultdict(dict)\n    for score, (qid, document) in zip(values, metadata, strict=True):\n        output[qid][document] = float(score)\n    return dict(output)\n\n\ndef protected_predictions(\n    blended_scores: np.ndarray,\n    metadata: list[tuple[str, str]],\n    baseline: dict[str, list[str]],\n    margin: float,\n) -> tuple[dict[str, list[str]], dict[str, int]]:\n    per_query = scores_by_question(blended_scores, metadata)\n    output: dict[str, list[str]] = {}\n    promoted_questions = 0\n    promotions = 0\n    for qid, scores in per_query.items():\n        selected = list(baseline[qid])\n        outsiders = [doc for doc, _ in sorted(scores.items(), key=lambda item: (-item[1], item[0])) if doc not in selected]\n        changed = False\n        for outsider in outsiders:\n            weakest = min(selected, key=lambda doc: (scores[doc], doc))\n            if scores[outsider] < scores[weakest] + margin:\n                break\n            selected[selected.index(weakest)] = outsider\n            promotions += 1\n            changed = True\n        if changed:\n            promoted_questions += 1\n        # Ordering is irrelevant to Recall, but sorting makes the output deterministic.\n        output[qid] = sorted(selected, key=lambda doc: (-scores[doc], doc))\n    return output, {"promoted_questions": promoted_questions, "promotions": promotions}\n\n\ndef new_model(c_value: float):\n    return make_pipeline(\n        StandardScaler(),\n        LogisticRegression(\n            C=c_value,\n            class_weight="balanced",\n            max_iter=600,\n            solver="liblinear",\n            random_state=2026,\n        ),\n    )\n\n\ndef baseline_predictions(\n    questions: list[dict[str, Any]],\n    fused: dict[str, dict[str, Any]],\n    rerankings: dict[str, dict[str, list[str]]],\n    final_weights: dict[str, Any],\n) -> dict[str, list[str]]:\n    return {\n        question["qid"]: rrf(\n            {\n                "first_stage": fused[question["qid"]]["candidates"],\n                "jina": rerankings["jina"][question["qid"]],\n                "vietnamese_reranker": rerankings["vietnamese_reranker"][question["qid"]],\n            },\n            final_weights["weights"],\n            final_weights["rrf_k"],\n            5,\n        )\n        for question in questions\n    }\n\n\ndef main() -> None:\n    work = Path(sys.argv[1])\n    config = yaml.safe_load(Path(sys.argv[2]).read_text(encoding="utf-8"))\n    artifacts = Path(config["paths"]["artifacts_dir"])\n    if not artifacts.is_absolute():\n        artifacts = work / artifacts\n\n    first_stage = read_json(artifacts / "first_stage_weights.json")\n    phase2_final = read_json(artifacts / "phase2_final_weights.json")\n    retrieval_train = read_json(artifacts / "retrieval_train.json")\n    retrieval_public = read_json(artifacts / "retrieval_public.json")\n    fused_limit = int(config["retrieval"]["fused_top_k"])\n    fused_train = fuse_cached(retrieval_train, first_stage["weights"], first_stage["rrf_k"], fused_limit)\n    fused_public = fuse_cached(retrieval_public, first_stage["weights"], first_stage["rrf_k"], fused_limit)\n    train_questions = list(read_jsonl(artifacts / "train_questions.jsonl"))\n    public_questions = list(read_jsonl(artifacts / "public_questions.jsonl"))\n    outer_folds = grouped_folds(train_questions, config["validation"]["folds"])\n    fold_questions = [question for question in train_questions if outer_folds[question["qid"]] == 0]\n\n    train_rerankings = {name: engine_ranks(artifacts, "train", name, 0) for name in RERANKERS}\n    public_rerankings = {name: engine_ranks(artifacts, "public", name) for name in RERANKERS}\n    for questions, rerankings, split in (\n        (fold_questions, train_rerankings, "train"),\n        (public_questions, public_rerankings, "public"),\n    ):\n        expected = {question["qid"] for question in questions}\n        for name, values in rerankings.items():\n            missing = expected.difference(values)\n            wrong_depth = [qid for qid in expected.intersection(values) if len(values[qid]) != 80]\n            if missing or wrong_depth:\n                raise RuntimeError(f"{split}/{name}: missing={len(missing)}, non_top80={len(wrong_depth)}")\n\n    baseline = baseline_predictions(fold_questions, fused_train, train_rerankings, phase2_final)\n    baseline_metrics = score_predictions(baseline, fold_questions)\n    if any(not set(baseline[q["qid"]]).issubset(fused_train[q["qid"]]["candidates"][:80]) for q in fold_questions):\n        raise RuntimeError("Phase 2 baseline selected a document outside top 80")\n\n    x_train, y_train, train_metadata, inner_groups = make_rows(\n        fold_questions, fused_train, retrieval_train, train_rerankings, True\n    )\n    if y_train.sum() == 0 or set(inner_groups) != set(range(5)):\n        raise RuntimeError(\n            f"Invalid training sample: positives={int(y_train.sum())}, inner_folds={sorted(set(inner_groups))}"\n        )\n    anchor_train = standardize_per_query(x_train[:, -2].astype(np.float64), train_metadata)\n    baseline_per_inner = {\n        str(fold): score_predictions(\n            baseline,\n            [question for question in fold_questions if inner_fold(question["question"]) == fold],\n        )\n        for fold in range(5)\n    }\n\n    trials: list[dict[str, Any]] = []\n    for c_value in (0.03, 0.1, 0.3, 1.0, 3.0):\n        oof = np.zeros(len(y_train), dtype=np.float64)\n        for heldout in range(5):\n            train_mask = inner_groups != heldout\n            valid_mask = inner_groups == heldout\n            model = new_model(c_value)\n            model.fit(x_train[train_mask], y_train[train_mask])\n            oof[valid_mask] = model.decision_function(x_train[valid_mask])\n        learned = standardize_per_query(oof, train_metadata)\n        for alpha in (0.35, 0.5, 0.65, 0.8, 1.0):\n            blended = alpha * learned + (1.0 - alpha) * anchor_train\n            for margin in (0.0, 0.25, 0.5, 0.75, 1.0):\n                prediction, promotion = protected_predictions(blended, train_metadata, baseline, margin)\n                per_inner = {\n                    str(fold): score_predictions(\n                        prediction,\n                        [question for question in fold_questions if inner_fold(question["question"]) == fold],\n                    )\n                    for fold in range(5)\n                }\n                deltas = [per_inner[str(fold)]["recall"] - baseline_per_inner[str(fold)]["recall"] for fold in range(5)]\n                trials.append(\n                    {\n                        "C": c_value,\n                        "alpha": alpha,\n                        "promotion_margin": margin,\n                        "metrics": score_predictions(prediction, fold_questions),\n                        "per_inner": per_inner,\n                        "nonnegative_inner_folds": sum(delta >= -1e-12 for delta in deltas),\n                        "worst_inner_recall_delta": min(deltas),\n                        **promotion,\n                    }\n                )\n\n    robust = [\n        row\n        for row in trials\n        if row["nonnegative_inner_folds"] >= 4 and row["worst_inner_recall_delta"] >= -0.005\n    ]\n    pool = robust or trials\n    best = max(\n        pool,\n        key=lambda row: (\n            row["metrics"]["recall"],\n            row["nonnegative_inner_folds"],\n            row["worst_inner_recall_delta"],\n            row["metrics"]["precision"],\n            -row["promotions"],\n            -row["C"],\n        ),\n    )\n\n    final_model = new_model(best["C"])\n    final_model.fit(x_train, y_train)\n    x_public, _, public_metadata, _ = make_rows(\n        public_questions, fused_public, retrieval_public, public_rerankings, False\n    )\n    learned_public = standardize_per_query(final_model.decision_function(x_public), public_metadata)\n    anchor_public = standardize_per_query(x_public[:, -2].astype(np.float64), public_metadata)\n    baseline_public = baseline_predictions(public_questions, fused_public, public_rerankings, phase2_final)\n    blended_public = best["alpha"] * learned_public + (1.0 - best["alpha"]) * anchor_public\n    final_prediction, private_promotion = protected_predictions(\n        blended_public, public_metadata, baseline_public, best["promotion_margin"]\n    )\n\n    corpus_ids = {row["doc_id"] for row in read_jsonl(artifacts / "corpus.jsonl")}\n    submission = {qid: {"answer": documents} for qid, documents in final_prediction.items()}\n    validate_submission_shape(submission, public_questions, corpus_ids)\n    submission_path = work / "submission_phase4_private_final.json"\n    write_json(submission_path, submission)\n    with (work / "phase4_blender.pkl").open("wb") as handle:\n        pickle.dump(final_model, handle)\n\n    report = {\n        "experiment_id": "phase4-phase2-legal-bge-protected-blender",\n        "training_questions": len(fold_questions),\n        "training_pairs": len(y_train),\n        "positive_pairs": int(y_train.sum()),\n        "feature_count": int(x_train.shape[1]),\n        "inner_fold_counts": {\n            str(fold): int(sum(inner_fold(question["question"]) == fold for question in fold_questions))\n            for fold in range(5)\n        },\n        "candidate_top_k": 80,\n        "candidate_recall_fold0": score_candidates(\n            {question["qid"]: fused_train[question["qid"]]["candidates"][:80] for question in fold_questions},\n            fold_questions,\n        )["candidate_recall"],\n        "phase2_baseline_fold0": baseline_metrics,\n        "phase2_baseline_per_inner": baseline_per_inner,\n        "selected_cross_fitted": best,\n        "recall_delta": best["metrics"]["recall"] - baseline_metrics["recall"],\n        "robust_trials": len(robust),\n        "tested_configurations": len(trials),\n        "private_promotions": private_promotion,\n        "submission": str(submission_path),\n    }\n    write_json(work / "phase4_report.json", report)\n    print(json.dumps(report, ensure_ascii=False, indent=2))\n\n\nif __name__ == "__main__":\n    main()', encoding='utf-8')
(WORK / 'phase41_ensemble.py').write_text('"""One cache-only Phase 4.1 candidate; never runs retrieval or reranking."""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport sys\nimport zipfile\nfrom pathlib import Path\n\nimport numpy as np\nimport yaml\n\nfrom legalir.storage import read_json, read_jsonl, write_json\nfrom legalir.text import clean_text\nfrom legalir.validation import grouped_folds, score_candidates, score_predictions, validate_submission_shape\nfrom phase4_blend import (\n    RERANKERS,\n    baseline_predictions,\n    engine_ranks,\n    fuse_cached,\n    inner_fold,\n    make_rows,\n    new_model,\n    protected_predictions,\n    standardize_per_query,\n)\n\n\nCS = (0.03, 0.1, 0.3)\nREPORT_NAME = "phase41_report.json"\nSUBMISSION_NAME = "submission_phase41_private_ensemble.json"\n\n\ndef required(path: Path) -> Path:\n    if not path.is_file() or path.stat().st_size == 0:\n        raise FileNotFoundError(f"Required cache/output missing or broken symlink: {path}")\n    return path\n\n\ndef verify_source(report: dict, private_path: Path, phase4_dir: Path, phase2_dir: Path) -> None:\n    content = required(private_path).read_bytes()\n    if (report.get("experiment_id") != "phase4-phase2-legal-bge-protected-blender"\n            or report.get("test_file") != "private-official.json"\n            or report.get("test_sha256") != hashlib.sha256(content).hexdigest()\n            or report.get("test_questions") != len(json.loads(content))):\n        raise RuntimeError("Phase 4 report does not match the official private questions")\n    p4 = read_json(required(phase4_dir / "artifacts_phase4" / "prepare_manifest.json"))\n    p2 = read_json(required(phase2_dir / "prepare_manifest.json"))\n    for key in ("schema_version", "chunking_fingerprint", "documents", "short_chunks", "long_chunks",\n                "train_questions", "public_questions", "train_questions_fingerprint", "public_questions_fingerprint"):\n        if p4.get(key) != p2.get(key):\n            raise RuntimeError(f"Phase 2/4 prepared data differs: {key}")\n    # Fail before training when a saved output omitted any expensive cache.\n    for name in ("first_stage_weights.json", "final_weights.json", "retrieval_train.json", "retrieval_public.json"):\n        required(phase2_dir / name)\n    for split, suffix in (("train", "_0"), ("public", "")):\n        for engine in ("jina", "vietnamese_reranker"):\n            required(phase2_dir / f"rerank_{split}{suffix}_{engine}.json")\n        required(phase4_dir / "artifacts_phase4" / f"rerank_{split}{suffix}_legal_reranker.json")\n    for name in ("train_questions.jsonl", "public_questions.jsonl", "corpus.jsonl"):\n        required(phase4_dir / "artifacts_phase4" / name)\n    for name in ("kaggle_rtx_pro_6000_phase4.yaml", "submission_phase4_private_final.json"):\n        required(phase4_dir / name)\n\n\ndef per_inner(predictions: dict[str, list[str]], questions: list[dict]) -> dict[str, dict[str, float]]:\n    return {\n        str(fold): score_predictions(\n            predictions, [q for q in questions if inner_fold(q["question"]) == fold]\n        ) for fold in range(5)\n    }\n\n\ndef gate(ensemble: dict, phase4: dict, tolerance: float = 1e-12) -> tuple[bool, str]:\n    if ensemble["metrics"]["recall"] <= phase4["metrics"]["recall"] + tolerance:\n        return False, "Ensemble OOF Recall did not exceed Phase 4"\n    for fold in range(5):\n        key = str(fold)\n        if ensemble["per_inner"][key]["recall"] < phase4["per_inner"][key]["recall"] - tolerance:\n            return False, f"Ensemble OOF Recall declined in inner fold {fold}"\n    return True, "OOF Recall improved with no declining inner fold"\n\n\ndef oof_scores(x: np.ndarray, y: np.ndarray, groups: np.ndarray, metadata: list[tuple[str, str]],\n               c_values: tuple[float, ...] = CS) -> dict[float, np.ndarray]:\n    scores = {}\n    for c in c_values:\n        raw = np.empty(len(y), dtype=np.float64)\n        for fold in range(5):\n            train, valid = groups != fold, groups == fold\n            if not valid.any() or len(np.unique(y[train])) != 2:\n                raise RuntimeError(f"Missing inner fold or class: {fold}")\n            model = new_model(c)\n            model.fit(x[train], y[train])\n            raw[valid] = model.decision_function(x[valid])\n        scores[c] = standardize_per_query(raw, metadata)\n    return scores\n\n\ndef compare_phase4(predictions: dict[str, list[str]], saved_submission: dict) -> int:\n    if set(predictions) != set(saved_submission):\n        raise RuntimeError("Saved Phase 4 submission QIDs differ from reconstructed QIDs")\n    changed = sum(set(documents) != set(saved_submission[qid]["answer"])\n                  for qid, documents in predictions.items())\n    return changed\n\n\ndef main(phase4_dir: Path, phase2_dir: Path, work: Path, private_path: Path) -> None:\n    work.mkdir(parents=True, exist_ok=True)\n    # A rerun must never leave an old report or submission masquerading as a new result.\n    for name in (REPORT_NAME, SUBMISSION_NAME, SUBMISSION_NAME.replace(".json", ".zip")):\n        stale = work / name\n        if stale.exists():\n            stale.unlink()\n    report = read_json(required(phase4_dir / "phase4_report.json"))\n    verify_source(report, private_path, phase4_dir, phase2_dir)\n    config = yaml.safe_load(required(phase4_dir / "kaggle_rtx_pro_6000_phase4.yaml").read_text(encoding="utf-8"))\n    if config["validation"]["folds"] != 5 or config["retrieval"]["fused_top_k"] < 80:\n        raise RuntimeError("Phase 4 fold count or candidate depth differs")\n    p4_artifacts = phase4_dir / "artifacts_phase4"\n    retrieval_train = read_json(phase2_dir / "retrieval_train.json")\n    retrieval_private = read_json(phase2_dir / "retrieval_public.json")\n    first_stage = read_json(phase2_dir / "first_stage_weights.json")\n    weights = read_json(phase2_dir / "final_weights.json")\n    fused_train = fuse_cached(retrieval_train, first_stage["weights"], first_stage["rrf_k"],\n                              config["retrieval"]["fused_top_k"])\n    fused_private = fuse_cached(retrieval_private, first_stage["weights"], first_stage["rrf_k"],\n                                config["retrieval"]["fused_top_k"])\n    train_questions = list(read_jsonl(p4_artifacts / "train_questions.jsonl"))\n    private_questions = list(read_jsonl(p4_artifacts / "public_questions.jsonl"))\n    official_private = json.loads(private_path.read_text(encoding="utf-8"))\n    if len(private_questions) != report["test_questions"] or len(official_private) != len(private_questions):\n        raise RuntimeError("Saved prepared private question count differs")\n    # Prepared questions and raw official input must have the same IDs and text.\n    if not isinstance(official_private, dict):\n        raise RuntimeError("Official private file must be a QID-to-question mapping")\n    if {q["qid"] for q in private_questions} != set(official_private) or any(\n        clean_text(official_private[q["qid"]]["question"]) != q["question"] for q in private_questions\n    ):\n        raise RuntimeError("Saved prepared private questions do not match official input")\n    fold_questions = [q for q in train_questions if grouped_folds([q], 5)[q["qid"]] == 0]\n    train_ranks = {name: engine_ranks(phase2_dir if name != "legal_reranker" else p4_artifacts,\n                                     "train", name, 0) for name in RERANKERS}\n    private_ranks = {name: engine_ranks(phase2_dir if name != "legal_reranker" else p4_artifacts,\n                                       "public", name) for name in RERANKERS}\n    for questions, retrieval, fused, ranks in (\n        (fold_questions, retrieval_train, fused_train, train_ranks),\n        (private_questions, retrieval_private, fused_private, private_ranks),\n    ):\n        for q in questions:\n            qid = q["qid"]\n            candidates = fused[qid]["candidates"][:80]\n            if len(candidates) != 80 or len(set(candidates)) != 80:\n                raise RuntimeError(f"Invalid Phase 4 top 80: {qid}")\n            for name in RERANKERS:\n                ranking = ranks[name][qid]\n                if len(ranking) != 80 or set(ranking) != set(candidates):\n                    raise RuntimeError(f"Phase 4 {name} ranking/top-80 mismatch: {qid}")\n            for name in ("bm25", "accent_char", "vietlegal_harrier", "vietnamese_embedding",\n                         "nemotron", "query_memory", "query_exact"):\n                if name not in retrieval[qid]["channels"]:\n                    raise RuntimeError(f"Missing retrieval channel {name}: {qid}")\n\n    baseline = baseline_predictions(fold_questions, fused_train, train_ranks, weights)\n    if abs(score_predictions(baseline, fold_questions)["recall"] -\n           report["phase2_baseline_fold0"]["recall"]) > 1e-10:\n        raise RuntimeError("Phase 2 baseline cannot be reproduced from caches")\n    x, y, metadata, groups = make_rows(fold_questions, fused_train, retrieval_train, train_ranks, True)\n    selected = report["selected_cross_fitted"]\n    if (selected["C"], selected["alpha"], selected["promotion_margin"]) != (0.1, 1.0, 0.0):\n        raise RuntimeError("Saved Phase 4 selected a different configuration")\n    if (len(fold_questions) != report["training_questions"] or len(y) != report["training_pairs"]\n            or int(y.sum()) != report["positive_pairs"] or x.shape[1] != report["feature_count"]\n            or score_candidates({q["qid"]: fused_train[q["qid"]]["candidates"][:80]\n                                 for q in fold_questions}, fold_questions)["candidate_recall"]\n            != report["candidate_recall_fold0"]):\n        raise RuntimeError("Reconstructed Phase 4 training data differs from report")\n    scores = oof_scores(x, y, groups, metadata)\n    phase4_pred, _ = protected_predictions(scores[0.1], metadata, baseline, 0.0)\n    ensemble_pred, promotions = protected_predictions(np.mean(list(scores.values()), axis=0), metadata, baseline, 0.0)\n    phase4_oof = {"metrics": score_predictions(phase4_pred, fold_questions),\n                  "per_inner": per_inner(phase4_pred, fold_questions)}\n    ensemble_oof = {"metrics": score_predictions(ensemble_pred, fold_questions),\n                    "per_inner": per_inner(ensemble_pred, fold_questions), **promotions}\n    if abs(phase4_oof["metrics"]["recall"] - selected["metrics"]["recall"]) > 1e-10 or any(\n        abs(phase4_oof["per_inner"][str(i)]["recall"] - selected["per_inner"][str(i)]["recall"]) > 1e-10\n        for i in range(5)\n    ):\n        raise RuntimeError("Phase 4 OOF cannot be reproduced; refusing to evaluate Phase 4.1")\n\n    approved, reason = gate(ensemble_oof, phase4_oof)\n    output = {"experiment_id": "phase41-cache-only-three-logistic-ensemble",\n              "phase4_report_sha256": hashlib.sha256((phase4_dir / "phase4_report.json").read_bytes()).hexdigest(),\n              "test_sha256": report["test_sha256"], "C_values": list(CS), "alpha": 1.0,\n              "promotion_margin": 0.0, "phase4_oof": phase4_oof, "ensemble_oof": ensemble_oof,\n              "oof_delta": ensemble_oof["metrics"]["recall"] - phase4_oof["metrics"]["recall"],\n              "approved": approved, "reason": reason, "submission": None,\n              "warning": "OOF selection is not an unbiased estimate of private Recall."}\n    if not approved:\n        write_json(work / REPORT_NAME, output)\n        print(json.dumps(output, ensure_ascii=False, indent=2))\n        return\n\n    x_private, _, private_meta, _ = make_rows(private_questions, fused_private,\n                                               retrieval_private, private_ranks, False)\n    if x_private.shape[1] != x.shape[1]:\n        raise RuntimeError("Private feature dimension differs from training")\n    full_scores = []\n    for c in CS:\n        model = new_model(c)\n        model.fit(x, y)\n        full_scores.append(standardize_per_query(model.decision_function(x_private), private_meta))\n    private_baseline = baseline_predictions(private_questions, fused_private, private_ranks, weights)\n    phase4_full = new_model(0.1)\n    phase4_full.fit(x, y)\n    reconstructed, _ = protected_predictions(\n        standardize_per_query(phase4_full.decision_function(x_private), private_meta),\n        private_meta, private_baseline, 0.0,\n    )\n    saved = read_json(phase4_dir / "submission_phase4_private_final.json")\n    if compare_phase4(reconstructed, saved):\n        raise RuntimeError("Reconstructed Phase 4 private answer sets differ from saved submission")\n    predictions, private_promotions = protected_predictions(\n        np.mean(full_scores, axis=0), private_meta, private_baseline, 0.0\n    )\n    changed = compare_phase4(predictions, saved)\n    if changed == 0:\n        output["approved"] = False\n        output["reason"] = "Ensemble is identical to Phase 4 on private inputs; do not spend last submission"\n        output["changed_answer_sets_vs_phase4"] = 0\n        write_json(work / REPORT_NAME, output)\n        print(json.dumps(output, ensure_ascii=False, indent=2))\n        return\n    corpus_ids = {row["doc_id"] for row in read_jsonl(p4_artifacts / "corpus.jsonl")}\n    submission = {qid: {"answer": docs} for qid, docs in predictions.items()}\n    validate_submission_shape(submission, private_questions, corpus_ids)\n    output["private_promotions"] = private_promotions\n    output["changed_answer_sets_vs_phase4"] = changed\n    output["submission"] = str(work / SUBMISSION_NAME)\n    output["zip"] = str(work / SUBMISSION_NAME.replace(".json", ".zip"))\n    write_json(work / SUBMISSION_NAME, submission)\n    with zipfile.ZipFile(output["zip"], "w", compression=zipfile.ZIP_DEFLATED) as handle:\n        handle.write(work / SUBMISSION_NAME, arcname=SUBMISSION_NAME)\n    write_json(work / REPORT_NAME, output)\n    print(json.dumps(output, ensure_ascii=False, indent=2))\n\n\nif __name__ == "__main__":\n    if len(sys.argv) != 5:\n        raise SystemExit("Usage: phase41_ensemble.py PHASE4_OUTPUT PHASE2_ARTIFACTS WORK_DIR PRIVATE_JSON")\n    main(*(Path(value) for value in sys.argv[1:]))', encoding='utf-8')
subprocess.run([sys.executable, str(WORK / 'phase41_ensemble.py'), str(PHASE4_OUTPUT),
                str(PHASE2_ARTIFACTS), str(WORK), str(PRIVATE_FILE)],
               check=True, env=runtime_env, cwd=WORK)
result = read(WORK / 'phase41_report.json')
if result['approved']:
    print('ELIGIBLE SINGLE SUBMISSION:', result['zip'])
else:
    print('DO NOT SPEND LAST SUBMISSION: retain Phase 4.', result['reason'])
